# 1D1V Vlasov–Poisson: dataset generation

Data-only extraction from `GRNO_MHD_and_VlasovPoisson_128 (1).ipynb`. Numerical solver implementations, physical parameters, split seeds, and original profile defaults are preserved. Neural models, training, checkpoint evaluation, and model-result plotting are omitted. Source cell numbers are recorded in each code cell's metadata.

The saved default is `PROFILE = "paper"`. Read the profile table in [README.md](README.md) before running: the `paper` profile can require substantial GPU time and disk space. Install a compatible PyTorch build plus NumPy, pandas, tqdm, and IPython in this notebook's kernel. These notebooks do not install packages automatically.

Run from the repository, or set the environment variable `GRNO_ROOT` to your intended project/data location before starting the kernel. The printed `ROOT` and data roots identify the generated files. Restart and run from the top after changing profiles. Set `RUN_DATA_GENERATION = False` to inspect a complete existing cache; simulator audits and normalization still run.

The source also contains an MHD extension. This extraction contains only `VPConfig`, `VlasovPoissonSolver`, the VP simulator audit, VP generation/cache validation, and the VP manifest branch. Setting `TASKS` alone in the original did not disable MHD generation; those unconditional MHD statements are explicitly removed here. The original solver uses FFT shifts along velocity; the manifest labels the finite velocity boundary nonperiodic. The original audit checks the small boundary tail. No boundary algorithm was changed. The released training reader computes VP normalization from training arrays with its own deterministic seed-17 estimator.


In [ ]:
from __future__ import annotations

import contextlib
import dataclasses
import hashlib
import json
import math
import os
import random
import shutil
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from IPython.display import display
from tqdm.auto import tqdm

# Storage location only: set GRNO_ROOT before starting this kernel to override it.
# No solver, split, seed, or physical parameter is changed by this root selection.
def locate_project_root() -> Path:
    override = os.environ.get("GRNO_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    return cwd

ROOT = locate_project_root()
ROOT.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print({"project_root": str(ROOT), "device": str(device), "torch": torch.__version__})


## Original VP profiles


In [ ]:
PROFILE = "paper"          # "smoke", "pilot", "paper"


RUN_DATA_GENERATION = True


DATASET_VERSION = "grno_mhd_vp_custom_pilot_v2"


@dataclass(frozen=True)
class RunConfig:
    profile: str
    n_train: int
    n_val: int
    n_test: int
    train_frames: int
    val_frames: int
    test_frames: int
    epochs: int
    samples_per_epoch: int
    batch_size: int
    train_rollout: int
    eval_rollout: int
    width: int
    depths: tuple
    lr: float
    weight_decay: float
    grad_weight: float
    physics_weight: float
    output_dir: str = "grno_mhd_vp_outputs"
    data_dir: str = "grno_mhd_vp_data"


@dataclass(frozen=True)
class VPConfig:
    name: str = "vlasov_poisson"
    nx: int = 128
    nv: int = 128
    Lx: float = 4.0 * math.pi
    vmax: float = 6.0
    frame_dt: float = 0.1
    substeps: int = 5
    landau_fraction: float = 0.5


if PROFILE == "smoke":
    RUN = RunConfig(PROFILE, 2, 1, 1, 7, 9, 11, 2, 24, 2, 1, 5,
                    16, (1, 1, 1), 1e-3, 1e-4, 0.03, 0.02)
elif PROFILE == "pilot":
    RUN = RunConfig(PROFILE, 20, 4, 6, 41, 61, 81, 40, 512, 4, 2, 20,
                    32, (2, 2, 3), 4e-4, 1e-4, 0.05, 0.05)
elif PROFILE == "paper":
    RUN = RunConfig(PROFILE, 64, 8, 12, 101, 121, 161, 100, 2048, 4, 4, 40,
                    48, (2, 3, 4), 3e-4, 1e-4, 0.05, 0.05)
else:
    raise ValueError(PROFILE)


VP = VPConfig()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


if PROFILE != "smoke" and device.type != "cuda":
    raise RuntimeError("pilot/paper are intended for one CUDA GPU; use smoke on CPU")


torch.backends.cuda.matmul.allow_tf32 = True


torch.backends.cudnn.allow_tf32 = True


TASKS = ("vlasov_poisson",)
assert VP.nx >= 128 and VP.nv >= 128
RUN = dataclasses.replace(RUN, data_dir=str(ROOT / RUN.data_dir))
print(asdict(RUN), asdict(VP), device, sep="\n")


In [ ]:
def short_hash(obj, n=10):
    payload = json.dumps(obj, sort_keys=True, default=str).encode()
    return hashlib.sha256(payload).hexdigest()[:n]


## Original split-spectral reference solver


In [ ]:
class VlasovPoissonSolver:
    def __init__(self,cfg,device):
        self.cfg,self.device=cfg,device
        self.x=torch.arange(cfg.nx,device=device)*cfg.Lx/cfg.nx
        self.v=-cfg.vmax+torch.arange(cfg.nv,device=device)*(2*cfg.vmax/cfg.nv)
        self.dx=cfg.Lx/cfg.nx; self.dv=2*cfg.vmax/cfg.nv
        self.kx=2*math.pi*torch.fft.fftfreq(cfg.nx,d=self.dx,device=device)
        self.kv=2*math.pi*torch.fft.fftfreq(cfg.nv,d=self.dv,device=device)
        self.poisson_active=self.kx!=0
        if cfg.nx % 2 == 0:
            self.poisson_active[cfg.nx//2]=False

    def electric(self,f):
        rho=f.sum(dim=-2)*self.dv
        rho_hat=torch.fft.fft(rho,dim=-1)
        E_hat=torch.zeros_like(rho_hat)
        nz=self.poisson_active
        # d_x E = 1-rho -> E_hat = i*rho_hat/k for k != 0.
        E_hat[:,nz]=1j*rho_hat[:,nz]/self.kx[nz]
        return torch.fft.ifft(E_hat,dim=-1).real

    def x_shift(self,f,dt):
        fh=torch.fft.fft(f,dim=-1)
        phase=torch.exp(-1j*self.v[None,:,None]*self.kx[None,None,:]*dt)
        return torch.fft.ifft(fh*phase,dim=-1).real

    def v_shift(self,f,E,dt):
        # f_t-E f_v=0 -> f_new(v)=f_old(v+E dt).
        fh=torch.fft.fft(f,dim=-2)
        phase=torch.exp(1j*self.kv[None,:,None]*E[:,None,:]*dt)
        return torch.fft.ifft(fh*phase,dim=-2).real

    def step(self,f,dt):
        f=self.x_shift(f,.5*dt)
        E=self.electric(f)
        f=self.v_shift(f,E,dt)
        return self.x_shift(f,.5*dt)

    def initial(self,batch,seed):
        rng=np.random.default_rng(seed)
        out=[]; labels=[]
        X=self.x[None,:]; V=self.v[:,None]
        for i in range(batch):
            landau=bool(rng.random()<self.cfg.landau_fraction)
            phase=float(rng.uniform(0,2*math.pi))
            if landau:
                alpha=float(rng.uniform(.05,.18)); sigma=float(rng.uniform(.8,1.2))
                base=torch.exp(-.5*(V/sigma)**2)/(math.sqrt(2*math.pi)*sigma)
                perturb=1+alpha*torch.cos(.5*X+phase)+.25*alpha*torch.cos(X-2*phase)
                f=base*perturb; labels.append(0)
            else:
                alpha=float(rng.uniform(.03,.10)); sigma=float(rng.uniform(.35,.60)); v0=float(rng.uniform(1.6,2.4))
                base=.5*(torch.exp(-.5*((V-v0)/sigma)**2)+torch.exp(-.5*((V+v0)/sigma)**2))/(math.sqrt(2*math.pi)*sigma)
                perturb=1+alpha*torch.cos(.5*X+phase)+.2*alpha*torch.cos(1.5*X-phase)
                f=base*perturb; labels.append(1)
            f=f/(f.sum()*self.dv*self.dx/self.cfg.Lx)
            out.append(f)
        return torch.stack(out),torch.tensor(labels,device=self.device)

    @torch.no_grad()
    def generate(self,path,regime_path,ntraj,frames,seed,batch=4):
        path,regime_path=Path(path),Path(regime_path)
        partial=path.with_name(path.name+".partial")
        regime_partial=regime_path.with_name(regime_path.name+".partial")
        partial.unlink(missing_ok=True); regime_partial.unlink(missing_ok=True)
        arr=np.lib.format.open_memmap(partial,mode="w+",dtype="float32",
            shape=(ntraj,frames,1,self.cfg.nv,self.cfg.nx))
        regimes=np.lib.format.open_memmap(regime_partial,mode="w+",dtype="int64",shape=(ntraj,))
        try:
            dt=self.cfg.frame_dt/self.cfg.substeps
            for start in tqdm(range(0,ntraj,batch),desc=f"VP {path.stem}"):
                count=min(batch,ntraj-start)
                f,label=self.initial(count,seed+start)
                arr[start:start+count,0,0]=f.cpu().numpy(); regimes[start:start+count]=label.cpu().numpy()
                for t in range(1,frames):
                    for _ in range(self.cfg.substeps): f=self.step(f,dt)
                    if not torch.isfinite(f).all(): raise FloatingPointError(f"VP nonfinite frame {t}")
                    arr[start:start+count,t,0]=f.cpu().numpy()
            arr.flush(); regimes.flush(); del arr,regimes
            # Replace the state last: its presence is the completion signal.
            os.replace(regime_partial,regime_path); os.replace(partial,path)
        except BaseException:
            with contextlib.suppress(Exception): del arr
            with contextlib.suppress(Exception): del regimes
            partial.unlink(missing_ok=True); regime_partial.unlink(missing_ok=True)
            raise


## VP simulator audit


In [ ]:
@torch.no_grad()
def simulator_preflight():
    v=VlasovPoissonSolver(VP,device); f,_=v.initial(4,202)
    mass0=f.sum((-2,-1))*v.dx*v.dv
    for _ in range(VP.substeps): f=v.step(f,VP.frame_dt/VP.substeps)
    mass1=f.sum((-2,-1))*v.dx*v.dv
    boundary=float(f[:,[0,1,-2,-1]].abs().max())
    negative=float((-f.clamp_max(0)).sum()/f.abs().sum().clamp_min(1e-9))
    result = {"vp_mass_rel_max": float(((mass1-mass0)/mass0).abs().max()),
              "vp_velocity_boundary_max": boundary, "vp_negative_mass_fraction": negative}
    print(json.dumps(result, indent=2))
    assert result["vp_mass_rel_max"] < 2e-4 and boundary < 2e-3 and negative < 2e-3
    return result

SIMULATOR_PREFLIGHT = simulator_preflight()


## Generate/cache VP arrays, regime labels, and manifests


In [ ]:
def dataset_paths(name, cfg):
    fingerprint = short_hash({'version': DATASET_VERSION, 'pde': asdict(cfg), 'split': {'train': [RUN.n_train, RUN.train_frames], 'val': [RUN.n_val, RUN.val_frames], 'test': [RUN.n_test, RUN.test_frames]}})
    root = Path(RUN.data_dir) / f'{name}_{PROFILE}_{fingerprint}'
    root.mkdir(parents=True, exist_ok=True)
    return {'root': root, 'manifest': root / 'manifest.json', **{f'{s}_states': root / f'{s}_states.npy' for s in ('train', 'val', 'test')}, **{f'{s}_regime': root / f'{s}_regime.npy' for s in ('train', 'val', 'test')}}
PATHS = {'vlasov_poisson': dataset_paths('vlasov_poisson', VP)}
SPLITS = {'train': (RUN.n_train, RUN.train_frames, 1103), 'val': (RUN.n_val, RUN.val_frames, 2207), 'test': (RUN.n_test, RUN.test_frames, 3301)}

def state_cache_valid(path, expected_shape):
    """Validate an atomically completed state cache, including every trajectory endpoint."""
    try:
        a = np.load(path, mmap_mode='r')
        if a.shape != tuple(expected_shape) or a.dtype != np.dtype('float32'):
            return False
        probe = np.asarray(a[:, [0, -1]])
        if not np.isfinite(probe).all():
            return False
        return bool(np.all(np.any(np.abs(probe).reshape(a.shape[0], -1) > 0, axis=1)))
    except (OSError, ValueError, EOFError):
        return False

def regime_cache_valid(path, ntraj):
    try:
        a = np.load(path, mmap_mode='r')
        return bool(a.shape == (ntraj,) and a.dtype == np.dtype('int64') and np.isin(np.asarray(a), [0, 1]).all())
    except (OSError, ValueError, EOFError):
        return False
if RUN_DATA_GENERATION:
    vsolver = VlasovPoissonSolver(VP, device)
    for split, (number, frames, seed) in SPLITS.items():
        vpath = PATHS['vlasov_poisson'][f'{split}_states']
        rpath = PATHS['vlasov_poisson'][f'{split}_regime']
        if not (state_cache_valid(vpath, (number, frames, 1, VP.nv, VP.nx)) and regime_cache_valid(rpath, number)):
            vsolver.generate(PATHS['vlasov_poisson'][f'{split}_states'], PATHS['vlasov_poisson'][f'{split}_regime'], number, frames, seed)
for split, (number, frames, _) in SPLITS.items():
    if not state_cache_valid(PATHS['vlasov_poisson'][f'{split}_states'], (number, frames, 1, VP.nv, VP.nx)):
        raise RuntimeError(f'invalid or incomplete VP state cache for {split}; enable RUN_DATA_GENERATION')
    if not regime_cache_valid(PATHS['vlasov_poisson'][f'{split}_regime'], number):
        raise RuntimeError(f'invalid or incomplete VP regime cache for {split}; enable RUN_DATA_GENERATION')
for name, cfg in (('vlasov_poisson', VP),):
    manifest = {'dataset_version': DATASET_VERSION, 'pde': asdict(cfg), 'run': asdict(RUN), 'state_layout': '[trajectory,time,channel,y_or_v,x]', 'state_channels': ['f'], 'axis_names': ['v', 'x'], 'periodic_axes': [False, True], 'domain_bounds': [[-VP.vmax, VP.vmax], [0.0, VP.Lx]], 'representation': 'phase-space density', 'files': {k: str(v) for k, v in PATHS[name].items() if k != 'root' and ('regime' not in k or name == 'vlasov_poisson')}}
    PATHS[name]['manifest'].write_text(json.dumps(manifest, indent=2))
    for split in SPLITS:
        arr = np.load(PATHS[name][f'{split}_states'], mmap_mode='r')
        print(name, split, arr.shape, round(arr.nbytes / 2 ** 30, 3), 'GiB', bool(np.isfinite(arr[0]).all()))
